### Load the thermal features

In [34]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path(
    r"C:\Users\DIPANSHU SHUKLA\OneDrive\Desktop\SIH_162_ML"
)

EVENT_FILE = (
    PROJECT_ROOT
    / "data"
    / "events"
    / "thermal_events_prototype.csv"
)

CONTEXT_FILE = (
    PROJECT_ROOT
    / "data"
    / "features"
    / "firms_context_features_time_aware.csv"
)

ANOMALY_FILE = (
    PROJECT_ROOT
    / "data"
    / "results"
    / "thermal_anomaly_prototype.csv"
)

events = pd.read_csv(EVENT_FILE)
context = pd.read_csv(CONTEXT_FILE)
anomaly = pd.read_csv(ANOMALY_FILE)

print("Events:", events.shape)
print("Context:", context.shape)
print("Anomaly:", anomaly.shape)

Events: (439251, 14)
Context: (702968, 9)
Anomaly: (439251, 25)


In [48]:
# ============================================================
# PREPARE EVENTS, ANOMALY AND CONTEXT
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# EVENTS
# ------------------------------------------------------------

events["acq_date"] = pd.to_datetime(
    events["acq_date"],
    errors="coerce"
)

# ------------------------------------------------------------
# ANOMALY
# ------------------------------------------------------------

anomaly["acq_date"] = pd.to_datetime(
    anomaly["acq_date"],
    errors="coerce"
)

# ------------------------------------------------------------
# CONTEXT
# ------------------------------------------------------------

context["timestamp"] = pd.to_datetime(
    context["timestamp"],
    errors="coerce"
)

context["acq_date"] = context["timestamp"].dt.normalize()

# ------------------------------------------------------------
# RECREATE SAME SPATIAL GRID
# ------------------------------------------------------------

SPATIAL_CELL_KM = 1.0

LAT_CELL = SPATIAL_CELL_KM / 111.0
LON_CELL = SPATIAL_CELL_KM / 100.0

context["lat_cell"] = np.floor(
    context["latitude"] / LAT_CELL
).astype("int32")

context["lon_cell"] = np.floor(
    context["longitude"] / LON_CELL
).astype("int32")

print("Events:", events.shape)
print("Anomaly:", anomaly.shape)
print("Context:", context.shape)

print("\nContext prepared:")
display(
    context[
        [
            "acq_date",
            "latitude",
            "longitude",
            "lat_cell",
            "lon_cell"
        ]
    ].head()
)

Events: (439251, 14)
Anomaly: (439251, 25)
Context: (702968, 12)

Context prepared:


,acq_date,latitude,longitude,lat_cell,lon_cell
0,2025-01-01,26.9361,94.0335,2989,9403
1,2025-01-01,25.3231,91.6008,2810,9160
2,2025-01-01,26.2760,92.8154,2916,9281
3,2025-01-01,25.3249,91.5900,2811,9159
4,2025-01-01,26.7587,92.7613,2970,9276


In [49]:
print(
    "Event grid example:"
)

display(
    events[
        [
            "acq_date",
            "latitude",
            "longitude",
            "lat_cell",
            "lon_cell"
        ]
    ].head()
)

print("\nContext grid example:")

display(
    context[
        [
            "acq_date",
            "latitude",
            "longitude",
            "lat_cell",
            "lon_cell"
        ]
    ].head()
)

Event grid example:


,acq_date,latitude,longitude,lat_cell,lon_cell
0,2025-01-01,8.984920,77.60124,997,7760
1,2025-01-01,9.000140,77.61005,999,7761
2,2025-01-01,9.278090,78.04165,1029,7804
3,2025-01-01,9.562105,77.63444,1061,7763
4,2025-01-01,9.630170,76.47209,1068,7647



Context grid example:


,acq_date,latitude,longitude,lat_cell,lon_cell
0,2025-01-01,26.9361,94.0335,2989,9403
1,2025-01-01,25.3231,91.6008,2810,9160
2,2025-01-01,26.2760,92.8154,2916,9281
3,2025-01-01,25.3249,91.5900,2811,9159
4,2025-01-01,26.7587,92.7613,2970,9276


In [50]:
# ============================================================
# AGGREGATE CONTEXT
# ============================================================

context_daily = (
    context
    .groupby(
        ["acq_date", "lat_cell", "lon_cell"],
        as_index=False
    )
    .agg(
        historical_detection_count=(
            "historical_detection_count",
            "max"
        ),

        historical_mean_frp=(
            "historical_mean_frp",
            "mean"
        ),

        local_frp_deviation=(
            "local_frp_deviation",
            "mean"
        ),

        historical_daily_activity=(
            "historical_daily_activity",
            "max"
        ),

        previously_detected=(
            "previously_detected",
            "max"
        )
    )
)

print("Context daily shape:", context_daily.shape)

print(
    "Duplicate context keys:",
    context_daily.duplicated(
        subset=[
            "acq_date",
            "lat_cell",
            "lon_cell"
        ]
    ).sum()
)

Context daily shape: (382881, 8)
Duplicate context keys: 0


In [35]:
# create event key
# ============================================================
# EVENT KEY
# ============================================================

event_key = [
    "acq_date",
    "start_time",
    "end_time",
    "latitude",
    "longitude"
]

for df_name, df in [
    ("events", events),
    ("anomaly", anomaly)
]:
    for col in event_key:
        if col in df.columns:
            print(df_name, col, "OK")

events acq_date OK
events start_time OK
events end_time OK
events latitude OK
events longitude OK
anomaly acq_date OK
anomaly start_time OK
anomaly end_time OK
anomaly latitude OK
anomaly longitude OK


In [51]:
# ============================================================
# EVENT + CONTEXT
# ============================================================

event_context = events.merge(
    context_daily,
    on=[
        "acq_date",
        "lat_cell",
        "lon_cell"
    ],
    how="left",
    validate="many_to_one"
)

print("Events:", len(events))
print("Event + Context:", len(event_context))

print(
    "Rows changed:",
    len(event_context) - len(events)
)

Events: 439251
Event + Context: 439251
Rows changed: 0


In [52]:
# ============================================================
# EVENT + CONTEXT + ANOMALY
# ============================================================

event_key = [
    "acq_date",
    "start_time",
    "end_time",
    "latitude",
    "longitude"
]

anomaly_selected = anomaly[
    event_key +
    [
        "duration_minutes",
        "frp_range",
        "frp_peak_ratio",
        "frp_per_observation",
        "start_hour",
        "day_of_week",
        "month",
        "is_weekend",
        "long_duration",
        "is_day",
        "is_night",
        "anomaly_score",
        "anomaly_prediction",
        "is_anomaly"
    ]
].copy()

ml_ready = event_context.merge(
    anomaly_selected,
    on=event_key,
    how="left",
    validate="one_to_one"
)

print("FINAL ML DATASET")
print("================")
print("Rows:", len(ml_ready))
print("Columns:", len(ml_ready.columns))

FINAL ML DATASET
Rows: 439251
Columns: 33


In [54]:
# ============================================================
# FINAL VALIDATION
# ============================================================

event_key = [
    "acq_date",
    "start_time",
    "end_time",
    "latitude",
    "longitude"
]

print("Total rows:", len(ml_ready))

print(
    "Unique event keys:",
    ml_ready[event_key].drop_duplicates().shape[0]
)

print(
    "Duplicate event keys:",
    ml_ready.duplicated(
        subset=event_key
    ).sum()
)

print(
    "Missing anomaly scores:",
    ml_ready["anomaly_score"].isna().sum()
)

print(
    "Missing anomaly labels:",
    ml_ready["is_anomaly"].isna().sum()
)

print("\nColumns:")
print(ml_ready.columns.tolist())

Total rows: 439251
Unique event keys: 439251
Duplicate event keys: 0
Missing anomaly scores: 382
Missing anomaly labels: 382

Columns:
['acq_date', 'lat_cell', 'lon_cell', 'event_number', 'start_time', 'end_time', 'observation_count', 'latitude', 'longitude', 'mean_frp', 'peak_frp', 'total_frp', 'mean_brightness', 'persistent', 'historical_detection_count', 'historical_mean_frp', 'local_frp_deviation', 'historical_daily_activity', 'previously_detected', 'duration_minutes', 'frp_range', 'frp_peak_ratio', 'frp_per_observation', 'start_hour', 'day_of_week', 'month', 'is_weekend', 'long_duration', 'is_day', 'is_night', 'anomaly_score', 'anomaly_prediction', 'is_anomaly']


In [55]:
# ============================================================
# SAVE CLEAN EVENT-LEVEL ML DATASET
# ============================================================

OUTPUT_FILE = (
    PROJECT_ROOT
    / "data"
    / "ml_ready"
    / "thermal_context_ml_ready_event_level.csv"
)

ml_ready.to_csv(
    OUTPUT_FILE,
    index=False
)

print("Saved successfully:")
print(OUTPUT_FILE)

Saved successfully:
C:\Users\DIPANSHU SHUKLA\OneDrive\Desktop\SIH_162_ML\data\ml_ready\thermal_context_ml_ready_event_level.csv


In [53]:
# ============================================================
# FINAL VALIDATION
# ============================================================

print("Total rows:", len(ml_ready))

print(
    "Unique event keys:",
    ml_ready[event_key].drop_duplicates().shape[0]
)

print(
    "Duplicate event keys:",
    ml_ready.duplicated(
        subset=event_key
    ).sum()
)

print(
    "Missing anomaly scores:",
    ml_ready["anomaly_score"].isna().sum()
)

print(
    "Missing anomaly labels:",
    ml_ready["is_anomaly"].isna().sum()
)

Total rows: 439251
Unique event keys: 439251
Duplicate event keys: 0
Missing anomaly scores: 382
Missing anomaly labels: 382


In [36]:
# ============================================================
# VERIFY ANOMALY ONE-TO-ONE
# ============================================================

assert len(anomaly) == anomaly[event_key].drop_duplicates().shape[0]

print("✅ Anomaly table has exactly one row per event.")

✅ Anomaly table has exactly one row per event.


In [38]:
print("Context columns:")
print(context.columns.tolist())

Context columns:
['latitude', 'longitude', 'timestamp', 'frp', 'historical_detection_count', 'historical_mean_frp', 'local_frp_deviation', 'historical_daily_activity', 'previously_detected']


In [39]:
# ============================================================
# PREPARE CONTEXT
# ============================================================

context["timestamp"] = pd.to_datetime(
    context["timestamp"],
    errors="coerce"
)

context["date_only"] = context["timestamp"].dt.normalize()

print("Context rows:", len(context))
print("Context columns:")
print(context.columns.tolist())

Context rows: 702968
Context columns:
['latitude', 'longitude', 'timestamp', 'frp', 'historical_detection_count', 'historical_mean_frp', 'local_frp_deviation', 'historical_daily_activity', 'previously_detected', 'date_only']


In [41]:
# ============================================================
# INSPECT CONTEXT DATASET
# ============================================================

print("Context shape:", context.shape)

print("\nContext columns:")
for i, col in enumerate(context.columns):
    print(i, repr(col))

print("\nFirst 5 rows:")
display(context.head())

print("\nData types:")
print(context.dtypes)

Context shape: (702968, 10)

Context columns:
0 'latitude'
1 'longitude'
2 'timestamp'
3 'frp'
4 'historical_detection_count'
5 'historical_mean_frp'
6 'local_frp_deviation'
7 'historical_daily_activity'
8 'previously_detected'
9 'date_only'

First 5 rows:


,latitude,longitude,timestamp,frp,historical_detection_count,historical_mean_frp,local_frp_deviation,historical_daily_activity,previously_detected,date_only
0,26.9361,94.0335,2025-01-01 03:50:00,6.5,0,0.0,6.5,0,0,2025-01-01
1,25.3231,91.6008,2025-01-01 03:51:00,8.0,0,0.0,8.0,0,0,2025-01-01
2,26.2760,92.8154,2025-01-01 03:51:00,11.6,0,0.0,11.6,0,0,2025-01-01
3,25.3249,91.5900,2025-01-01 03:51:00,11.3,0,0.0,11.3,0,0,2025-01-01
4,26.7587,92.7613,2025-01-01 03:51:00,5.3,0,0.0,5.3,0,0,2025-01-01



Data types:
latitude                             float64
longitude                            float64
timestamp                     datetime64[us]
frp                                  float64
historical_detection_count             int64
historical_mean_frp                  float64
local_frp_deviation                  float64
historical_daily_activity              int64
previously_detected                    int64
date_only                     datetime64[us]
dtype: object


In [47]:
print("EVENT columns:")
print(events.columns.tolist())

print("\nANOMALY columns:")
print(anomaly.columns.tolist())

print("\nCONTEXT columns:")
print(context.columns.tolist())

EVENT columns:
['acq_date', 'lat_cell', 'lon_cell', 'event_number', 'start_time', 'end_time', 'observation_count', 'latitude', 'longitude', 'mean_frp', 'peak_frp', 'total_frp', 'mean_brightness', 'persistent']

ANOMALY columns:
['acq_date', 'start_time', 'end_time', 'latitude', 'longitude', 'observation_count', 'duration_minutes', 'mean_frp', 'peak_frp', 'total_frp', 'frp_range', 'frp_peak_ratio', 'frp_per_observation', 'mean_brightness', 'start_hour', 'day_of_week', 'month', 'is_weekend', 'persistent', 'long_duration', 'is_day', 'is_night', 'anomaly_score', 'anomaly_prediction', 'is_anomaly']

CONTEXT columns:
['latitude', 'longitude', 'timestamp', 'frp', 'historical_detection_count', 'historical_mean_frp', 'local_frp_deviation', 'historical_daily_activity', 'previously_detected']


In [42]:
# ============================================================
# PREPARE EVENTS
# ============================================================

events["acq_date"] = pd.to_datetime(
    events["acq_date"],
    errors="coerce"
)

events["date_only"] = events["acq_date"].dt.normalize()

print(
    events[
        ["acq_date", "date_only", "lat_cell", "lon_cell"]
    ].head()
)

    acq_date  date_only  lat_cell  lon_cell
0 2025-01-01 2025-01-01       997      7760
1 2025-01-01 2025-01-01       999      7761
2 2025-01-01 2025-01-01      1029      7804
3 2025-01-01 2025-01-01      1061      7763
4 2025-01-01 2025-01-01      1068      7647


### Load time-aware context

In [2]:
CONTEXT_FILE = (
    PROJECT_ROOT
    / "data"
    / "features"
    / "firms_context_features_time_aware.csv"
)

context = pd.read_csv(CONTEXT_FILE)

print("Context features shape:", context.shape)
context.head()

Context features shape: (702968, 9)


,latitude,longitude,timestamp,frp,historical_detection_count,historical_mean_frp,local_frp_deviation,historical_daily_activity,previously_detected
0,26.9361,94.0335,2025-01-01 03:50:00,6.5,0,0.0,6.5,0,0
1,25.3231,91.6008,2025-01-01 03:51:00,8.0,0,0.0,8.0,0,0
2,26.2760,92.8154,2025-01-01 03:51:00,11.6,0,0.0,11.6,0,0
3,25.3249,91.5900,2025-01-01 03:51:00,11.3,0,0.0,11.3,0,0
4,26.7587,92.7613,2025-01-01 03:51:00,5.3,0,0.0,5.3,0,0


#### Inspect columns

In [3]:
print("THERMAL FEATURES")
print(thermal.columns.tolist())

print("\nCONTEXT FEATURES")
print(context.columns.tolist())

THERMAL FEATURES
['acq_date', 'start_time', 'end_time', 'latitude', 'longitude', 'observation_count', 'duration_minutes', 'mean_frp', 'peak_frp', 'total_frp', 'frp_range', 'frp_peak_ratio', 'frp_per_observation', 'mean_brightness', 'start_hour', 'day_of_week', 'month', 'is_weekend', 'persistent', 'long_duration', 'is_day', 'is_night']

CONTEXT FEATURES
['latitude', 'longitude', 'timestamp', 'frp', 'historical_detection_count', 'historical_mean_frp', 'local_frp_deviation', 'historical_daily_activity', 'previously_detected']


In [45]:
print("EVENT columns:")
print(events.columns.tolist())

print("\nANOMALY columns:")
print(anomaly.columns.tolist())

print("\nCONTEXT columns:")
print(context.columns.tolist())

EVENT columns:
['acq_date', 'lat_cell', 'lon_cell', 'event_number', 'start_time', 'end_time', 'observation_count', 'latitude', 'longitude', 'mean_frp', 'peak_frp', 'total_frp', 'mean_brightness', 'persistent']

ANOMALY columns:
['acq_date', 'start_time', 'end_time', 'latitude', 'longitude', 'observation_count', 'duration_minutes', 'mean_frp', 'peak_frp', 'total_frp', 'frp_range', 'frp_peak_ratio', 'frp_per_observation', 'mean_brightness', 'start_hour', 'day_of_week', 'month', 'is_weekend', 'persistent', 'long_duration', 'is_day', 'is_night', 'anomaly_score', 'anomaly_prediction', 'is_anomaly']

CONTEXT columns:
['latitude', 'longitude', 'timestamp', 'frp', 'historical_detection_count', 'historical_mean_frp', 'local_frp_deviation', 'historical_daily_activity', 'previously_detected']


#### common event identifier



In [4]:
thermal["start_time"] = pd.to_datetime(
    thermal["start_time"],
    errors="coerce"
)

context["timestamp"] = pd.to_datetime(
    context["timestamp"],
    errors="coerce"
)

thermal["acq_date"] = pd.to_datetime(
    thermal["acq_date"],
    errors="coerce"
)

#### Spatial cells in thermal data

In [5]:
SPATIAL_CELL_KM = 1.0

LAT_CELL = SPATIAL_CELL_KM / 111.0
LON_CELL = SPATIAL_CELL_KM / 100.0

thermal["lat_cell"] = (
    np.floor(
        thermal["latitude"] / LAT_CELL
    ).astype("int32")
)

thermal["lon_cell"] = (
    np.floor(
        thermal["longitude"] / LON_CELL
    ).astype("int32")
)

#### Create date key

In [6]:
thermal["date_only"] = (
    thermal["start_time"]
    .dt.normalize()
)

context["date_only"] = (
    context["timestamp"]
    .dt.normalize()
)

In [7]:
context["lat_cell"] = (
    np.floor(
        context["latitude"] / LAT_CELL
    ).astype("int32")
)

context["lon_cell"] = (
    np.floor(
        context["longitude"] / LON_CELL
    ).astype("int32")
)

#### Aggregate context to event-like level

In [9]:
## date + spatial cell

context_agg = (
    context
    .groupby(
        [
            "date_only",
            "lat_cell",
            "lon_cell"
        ],
        as_index=False
    )
    .agg(
        historical_detection_count=(
            "historical_detection_count",
            "max"
        ),
        historical_mean_frp=(
            "historical_mean_frp",
            "mean"
        ),
        local_frp_deviation=(
            "local_frp_deviation",
            "mean"
        ),
        historical_daily_activity=(
            "historical_daily_activity",
            "max"
        ),
        previously_detected=(
            "previously_detected",
            "max"
        )
    )
)

print("Aggregated context shape:", context_agg.shape)
context_agg.head()

Aggregated context shape: (382881, 8)


,date_only,lat_cell,lon_cell,historical_detection_count,historical_mean_frp,local_frp_deviation,historical_daily_activity,previously_detected
0,2025-01-01,997,7760,0,0.000,8.35,0,0
1,2025-01-01,999,7761,0,0.000,2.68,0,0
2,2025-01-01,1029,7804,0,0.000,1.58,0,0
3,2025-01-01,1061,7763,1,0.615,0.49,0,1
4,2025-01-01,1068,7647,0,0.000,2.11,0,0


### Merge thermal + context

In [10]:
combined = thermal.merge(
    context_agg,
    on=[
        "date_only",
        "lat_cell",
        "lon_cell"
    ],
    how="left",
    suffixes=("", "_context")
)

print("Combined shape:", combined.shape)
combined.head()

Combined shape: (439251, 30)


,acq_date,start_time,end_time,latitude,longitude,observation_count,duration_minutes,mean_frp,peak_frp,total_frp,...,is_day,is_night,lat_cell,lon_cell,date_only,historical_detection_count,historical_mean_frp,local_frp_deviation,historical_daily_activity,previously_detected
0,2025-01-01,2025-01-01 08:01:00,2025-01-01 08:01:00,8.984920,77.60124,1,0.0,8.350,8.35,8.35,...,1,0,997,7760,2025-01-01,0,0.000,8.35,0,0
1,2025-01-01,2025-01-01 08:01:00,2025-01-01 08:01:00,9.000140,77.61005,1,0.0,2.680,2.68,2.68,...,1,0,999,7761,2025-01-01,0,0.000,2.68,0,0
2,2025-01-01,2025-01-01 08:01:00,2025-01-01 08:01:00,9.278090,78.04165,1,0.0,1.580,1.58,1.58,...,1,0,1029,7804,2025-01-01,0,0.000,1.58,0,0
3,2025-01-01,2025-01-01 19:49:00,2025-01-01 19:49:00,9.562105,77.63444,2,0.0,1.105,1.23,2.21,...,0,1,1061,7763,2025-01-01,1,0.615,0.49,0,1
4,2025-01-01,2025-01-01 07:38:00,2025-01-01 07:38:00,9.630170,76.47209,1,0.0,2.110,2.11,2.11,...,1,0,1068,7647,2025-01-01,0,0.000,2.11,0,0


Check missing contextual values

In [12]:
context_columns = [
    "historical_detection_count",
    "historical_mean_frp",
    "local_frp_deviation",
    "historical_daily_activity",
    "previously_detected"
]

print(
    combined[context_columns].isna().sum()
)

historical_detection_count    0
historical_mean_frp           0
local_frp_deviation           0
historical_daily_activity     0
previously_detected           0
dtype: int64


In [13]:
# fill missing value
combined[context_columns] = (
    combined[context_columns]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

print("Missing values after cleaning:")
print(
    combined[context_columns].isna().sum()
)

Missing values after cleaning:
historical_detection_count    0
historical_mean_frp           0
local_frp_deviation           0
historical_daily_activity     0
previously_detected           0
dtype: int64


anomaly score

In [15]:
ANOMALY_FILE = (
    PROJECT_ROOT
    / "data"
    / "results"
    / "thermal_anomaly_prototype.csv"
)

anomaly = pd.read_csv(ANOMALY_FILE)

print("Anomaly data shape:", anomaly.shape)

Anomaly data shape: (439251, 25)


In [16]:
anomaly["start_time"] = pd.to_datetime(
    anomaly["start_time"],
    errors="coerce"
)

anomaly["date_only"] = (
    anomaly["start_time"].dt.normalize()
)

anomaly["lat_cell"] = (
    np.floor(
        anomaly["latitude"] / LAT_CELL
    ).astype("int32")
)

anomaly["lon_cell"] = (
    np.floor(
        anomaly["longitude"] / LON_CELL
    ).astype("int32")
)

In [17]:
anomaly_small = anomaly[
    [
        "date_only",
        "lat_cell",
        "lon_cell",
        "anomaly_score",
        "is_anomaly"
    ]
].copy()

In [18]:
#  merge
combined = combined.merge(
    anomaly_small,
    on=[
        "date_only",
        "lat_cell",
        "lon_cell"
    ],
    how="left"
)

print("Final combined shape:", combined.shape)

Final combined shape: (570717, 32)


##### Check the final dataset

In [19]:
print("Missing anomaly scores:")
print(
    combined["anomaly_score"].isna().sum()
)

print("\nMissing anomaly labels:")
print(
    combined["is_anomaly"].isna().sum()
)

Missing anomaly scores:
0

Missing anomaly labels:
0


#### Define ML features

In [20]:
thermal_features = [
    "observation_count",
    "duration_minutes",
    "mean_frp",
    "peak_frp",
    "total_frp",
    "frp_range",
    "frp_peak_ratio",
    "frp_per_observation",
    "mean_brightness",
    "start_hour",
    "day_of_week",
    "month",
    "is_weekend",
    "persistent",
    "long_duration",
    "is_day",
    "is_night"
]

context_features = [
    "historical_detection_count",
    "historical_mean_frp",
    "local_frp_deviation",
    "historical_daily_activity",
    "previously_detected"
]

model_features = (
    thermal_features
    + context_features
)

print("Number of ML features:", len(model_features))

Number of ML features: 22


Create X

In [21]:
X = combined[model_features].copy()

X = X.replace(
    [np.inf, -np.inf],
    np.nan
)

X = X.fillna(0)

print("X shape:", X.shape)
print("\nMissing values:")
print(X.isna().sum().sum())

X shape: (570717, 22)

Missing values:
0


##### Inspect correlations

In [22]:
correlation = X.corr()

print(
    correlation.round(2)
)

                            observation_count  duration_minutes  mean_frp  \
observation_count                        1.00              0.76      0.12   
duration_minutes                         0.76              1.00      0.03   
mean_frp                                 0.12              0.03      1.00   
peak_frp                                 0.24              0.12      0.90   
total_frp                                0.41              0.20      0.67   
frp_range                                0.33              0.20      0.45   
frp_peak_ratio                           0.73              0.65      0.17   
frp_per_observation                      0.12              0.03      1.00   
mean_brightness                          0.04             -0.02      0.25   
start_hour                               0.09              0.12     -0.20   
day_of_week                             -0.01             -0.00     -0.03   
month                                    0.05              0.03      0.10   

##### Save the ML-ready prototype

In [23]:
ML_DIR = (
    PROJECT_ROOT
    / "data"
    / "ml_ready"
)

ML_DIR.mkdir(
    parents=True,
    exist_ok=True
)

ML_FILE = (
    ML_DIR
    / "thermal_context_ml_ready_prototype.csv"
)

combined.to_csv(
    ML_FILE,
    index=False
)

print("Saved successfully:")
print(ML_FILE)
print("Shape:", combined.shape)

Saved successfully:
C:\Users\DIPANSHU SHUKLA\OneDrive\Desktop\SIH_162_ML\data\ml_ready\thermal_context_ml_ready_prototype.csv
Shape: (570717, 32)


In [24]:
print("Missing anomaly scores:")
print(combined["anomaly_score"].isna().sum())

print("Missing anomaly labels:")
print(combined["is_anomaly"].isna().sum())

Missing anomaly scores:
0
Missing anomaly labels:
0


verify the merged dataset


In [25]:
# ============================================================
# CELL 12 — CHECK MERGED DATASET
# ============================================================

print("Shape of combined dataset:", combined.shape)

print("\nDuplicate rows:")
print(combined.duplicated().sum())

print("\nMissing values:")
print(combined.isna().sum())

print("\nAnomaly label distribution:")
print(combined["is_anomaly"].value_counts())

print("\nAnomaly label percentage:")
print(combined["is_anomaly"].value_counts(normalize=True) * 100)

Shape of combined dataset: (570717, 32)

Duplicate rows:
0

Missing values:
acq_date                       0
start_time                     0
end_time                       0
latitude                       0
longitude                      0
observation_count              0
duration_minutes               0
mean_frp                       0
peak_frp                       0
total_frp                      0
frp_range                      0
frp_peak_ratio                35
frp_per_observation            0
mean_brightness                0
start_hour                     0
day_of_week                    0
month                          0
is_weekend                     0
persistent                     0
long_duration                  0
is_day                         0
is_night                       0
lat_cell                       0
lon_cell                       0
date_only                      0
historical_detection_count     0
historical_mean_frp            0
local_frp_deviation            0


In [26]:
# ============================================================
# CELL 14 — EVENT UNIQUENESS CHECK
# ============================================================

event_keys = [
    "acq_date",
    "start_time",
    "end_time",
    "latitude",
    "longitude"
]

print("Total rows:", len(combined))

unique_events = combined[event_keys].drop_duplicates()

print("Unique event keys:", len(unique_events))
print("Duplicated event rows:", len(combined) - len(unique_events))

if len(combined) == len(unique_events):
    print("\n✅ 1 ML row = 1 unique thermal event")
else:
    print("\n⚠️ Multiple rows are associated with the same thermal event")

Total rows: 570717
Unique event keys: 439251
Duplicated event rows: 131466

⚠️ Multiple rows are associated with the same thermal event


In [27]:
# ============================================================
# CELL 15 — HANDLE FRP RATIO
# ============================================================

print("Missing frp_peak_ratio before:",
      combined["frp_peak_ratio"].isna().sum())

combined["frp_peak_ratio"] = (
    combined["frp_peak_ratio"]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

print("Missing frp_peak_ratio after:",
      combined["frp_peak_ratio"].isna().sum())

Missing frp_peak_ratio before: 35
Missing frp_peak_ratio after: 0


In [28]:
# ============================================================
# CELL 16 — FEATURE LIST
# ============================================================

print("Number of columns:", len(combined.columns))

for i, col in enumerate(combined.columns, 1):
    print(f"{i:02d}. {col}")

Number of columns: 32
01. acq_date
02. start_time
03. end_time
04. latitude
05. longitude
06. observation_count
07. duration_minutes
08. mean_frp
09. peak_frp
10. total_frp
11. frp_range
12. frp_peak_ratio
13. frp_per_observation
14. mean_brightness
15. start_hour
16. day_of_week
17. month
18. is_weekend
19. persistent
20. long_duration
21. is_day
22. is_night
23. lat_cell
24. lon_cell
25. date_only
26. historical_detection_count
27. historical_mean_frp
28. local_frp_deviation
29. historical_daily_activity
30. previously_detected
31. anomaly_score
32. is_anomaly


In [29]:
# ============================================================
# CELL 17 — FINAL EVENT UNIQUENESS CHECK
# ============================================================

event_keys = [
    "acq_date",
    "start_time",
    "end_time",
    "latitude",
    "longitude"
]

total_rows = len(combined)
unique_events = combined[event_keys].drop_duplicates().shape[0]

print("Total ML rows:", total_rows)
print("Unique thermal events:", unique_events)
print("Extra duplicated event rows:", total_rows - unique_events)

if total_rows == unique_events:
    print("\n✅ CLEAN: 1 ML row = 1 thermal event")
else:
    print("\n⚠️ DUPLICATION FOUND: merge needs correction")

Total ML rows: 570717
Unique thermal events: 439251
Extra duplicated event rows: 131466

⚠️ DUPLICATION FOUND: merge needs correction


In [30]:
# ============================================================
# CELL 18 — CHECK FEATURE TYPES
# ============================================================

print("Numerical columns:")
print(combined.select_dtypes(include=np.number).columns.tolist())

print("\nNon-numerical columns:")
print(combined.select_dtypes(exclude=np.number).columns.tolist())

Numerical columns:
['latitude', 'longitude', 'observation_count', 'duration_minutes', 'mean_frp', 'peak_frp', 'total_frp', 'frp_range', 'frp_peak_ratio', 'frp_per_observation', 'mean_brightness', 'start_hour', 'day_of_week', 'month', 'is_weekend', 'persistent', 'long_duration', 'is_day', 'is_night', 'lat_cell', 'lon_cell', 'historical_detection_count', 'historical_mean_frp', 'local_frp_deviation', 'historical_daily_activity', 'previously_detected', 'anomaly_score', 'is_anomaly']

Non-numerical columns:
['acq_date', 'start_time', 'end_time', 'date_only']
